# Hva skjer når du øker tidssteget?

Vi følger fargestoff i et rør der vannet strømmer mot høyre med konstant hastighet. I den eksakte løsningen flytter startfunksjonen seg uten å endre form.

Første gang du åpner notatboka, kjører du de to første kodecellene én gang med **Skift+Enter**. Gå deretter til demonstrasjonen nederst. Når du endrer funksjon, steglengder eller visning, kjører du **bare den siste cellen**. Etter en omstart av kjernen kjører du oppsettet på nytt.

## Oppvindsmetoden

Ved positiv hastighet bruker vi verdien ved $x_j$ og verdien hos naboen til venstre, begge fra forrige tidsrad:
$$U_j^{n+1}=(1-\nu)U_j^n+\nu U_{j-1}^n, \qquad \nu=c\,\Delta t/\Delta x.$$
I koden er `u[n, j]` verdien $U_j^n$. Løkken beregner rad `n` fra rad `n - 1`. Funksjonen `oppvind` finner Courant-tallet fra avstandene mellom de første punktene i `x` og de første tidspunktene i `t`.

Vi fyller første rad med $f(x)$ og velger innløpet $u(0,t)=f(-ct)$. Da har vi den eksakte løsningen $u(x,t)=f(x-ct)$ å sammenlikne med.

In [ ]:
import numpy as np

def gitter(slutt, steg):
    if not 0 < steg <= slutt:
        raise ValueError("Steglengden må være positiv og høyst lik sluttverdien.")
    antall = int(np.nextafter(slutt / steg, np.inf))
    return steg * np.arange(antall + 1)

def oppvind(f, x, t, c):
    dx = x[1] - x[0]
    dt = t[1] - t[0]
    nu = c * dt / dx     # couranttallet 
    U = np.empty((t.size, x.size))
    U[0] = f(x)
    for n in range(1, t.size):
        U[n, 0] = f(-c * t[n])
        for j in range(1, x.size):
            U[n, j] = (1 - nu) * U[n - 1, j] + nu * U[n - 1, j - 1]
    return U

## Oppsett for visualisering

Denne cellen lager funksjonen som tegner den eksakte løsningen og gitterverdiene og viser dem som en animasjon. Kjør cellen én gang.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def vis_animasjon(f, x, t, u, c, nu, vis_linje=False):
    fig, ax = plt.subplots(figsize=(7.5, 3.6), constrained_layout=True)
    x_fint = np.linspace(0, 1, 1001)
    eksakt, = ax.plot(x_fint, f(x_fint), "--", color="tab:orange", label="Eksakt løsning")
    stil = "-" if vis_linje else "o"
    beregnet, = ax.plot(x, u[0], stil, color="tab:blue", markersize=4, label="Oppvind")
    ax.set(xlim=(0, 1), xlabel="Posisjon x ", ylabel="Konsentrasjon")
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1))

    def vis_tid(n):
        beregnet.set_ydata(u[n])
        eksakt.set_ydata(f(x_fint - c * t[n]))
        ax.set_title(f"t = {t[n]:.3f} s     ν = {nu:g}")
        ax.relim()
        ax.autoscale_view(scalex=False)

    bilder = np.linspace(0, t.size - 1, min(61, t.size), dtype=int)
    film = FuncAnimation(fig, vis_tid, frames=bilder, interval=100)
    plt.close(fig)
    return HTML(film.to_jshtml(default_mode="once"))

## Demonstrasjon

Velg funksjon, romsteg `dx` og tidssteg `dt` i cellen under. Trykk **Ctrl+Enter** etter hver endring for å kjøre cellen og bli stående i den. Cellen beregner Courant-tallet og løsningen på nytt og viser den nye animasjonen. Les av tiden i tittelen og tallene på den loddrette aksen.

Sett `vis_linje = True` for å forbinde de blå gitterverdiene med rette linjestykker, eller bruk `False` for å vise punktene. Valget endrer bare figuren, og den oransje kurven viser den eksakte løsningen i begge tilfeller.

1. Prøv gaussfunksjonen med `dx = 0.02` og `dt = 0.01`. Trykk på ▶. Hvor godt følger de blå gitterverdiene den skarpe kanten?
2. Behold romsteget og sett `dt = 0.03`. Hvilket Courant-tall får du? Når kommer den første verdien utenfor intervallet $[1,2]$?
3. Velg `f = trinnfunksjon` og prøv begge tidsstegene. Behold så `dt = 0.03` og prøv `dx = 0.04`. Forutsi hva som skjer før du kjører cellen.

In [ ]:
def trinnfunksjon(x):
    return np.where(x < 0.21, 2.0, 1.0)

def gaussfunksjon(x):
    return 1.0 + np.exp(-0.5 * ((x - 0.21) / 0.04)**2)

# Velg startfunksjon, steglengder og visning her.
f = gaussfunksjon        # trinnfunksjon eller gaussfunksjon
dx = 0.02                # romsteg 
dt = 0.01                # tidssteg 
c = 1.0                  # hastighet mot høyre
T = 0.60                 # sluttid
vis_linje = True        # True for linje, False for punkter

x = gitter(1.0, dx)
t = gitter(T, dt)
nu = c * dt / dx
u = oppvind(f, x, t, c)

print(f"Courant-tall ν = {nu:g}.")
vis_animasjon(f, x, t, u, c, nu, vis_linje=vis_linje)